# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all components by their `@id` field, as required by the Croissant standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print out the main description using metadata attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by `@id`.

In [ ]:
# List all record sets with their @id and details
print("Record Sets and their fields:")
record_set_ids = []
for rs in dataset.metadata.recordSet:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    print(f"- Record Set @id: {rs_id}")
    print(f"  Name: {rs.get('name','')}")
    print(f"  Description: {rs.get('description','')}")
    # List fields
    print("  Fields:")
    for fld in rs.get('field', []):
        if isinstance(fld, dict):
            fld_id = fld['@id']
        else:
            fld_id = fld
        # In Croissant, fields themselves are in dataset.metadata.field
        field_obj = next((f for f in dataset.metadata.field if f["@id"]==fld_id), None)
        if field_obj:
            print(f"    - Field @id: {fld_id}")
            print(f"      Name: {field_obj.get('name','')}")
            print(f"      DataType: {field_obj.get('dataType','')}")
        else:
            print(f"    - Field @id: {fld_id} (not found in global field list)")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references are via `@id`.

In [ ]:
# Collect all record set @id values
record_sets_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
dataframes = {}

# Load records for each record set using its @id
for record_set_id in record_sets_ids:
    # In mlcroissant, the record_set argument must be the '@id' of the RecordSet
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")

# Example: show columns of the first record set
if record_sets_ids:
    first_record_set_id = record_sets_ids[0]
    print(f"Columns for RecordSet @id {first_record_set_id}:\n", dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, categorization.

**Note**: Use the `@id` for the field name variables!

In [ ]:
# For demonstration, select the first record set and attempt to process numeric fields
record_set_id = record_sets_ids[0]
df = dataframes[record_set_id]

print(f"Checking for numeric fields in RecordSet: {record_set_id}")
# Find numeric fields by inspecting the data types in the dataframe
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields: {numeric_fields}")

if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Use the @id (the column name in DataFrame)
    threshold = df[numeric_field_id].mean()  # Use mean as example threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Find a categorical field to group by (example: field with type 'object' and not the numeric field)
    group_fields = [col for col in df.columns if (df[col].dtype == 'object') and (col != numeric_field_id)]
    if group_fields:
        group_field_id = group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric fields identified for this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. If possible, we use the field @id as columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Visualize the distribution of the numeric field
if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If group field exists, boxplot
    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} in {record_set_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In summary, we've demonstrated how to:
- Load and explore a Croissant-structured dataset using `mlcroissant`
- Access and analyze record sets, fields, and their IDs
- Perform exploratory data manipulation and normalization using @id references
- Visualize data distributions and group metrics

This workflow ensures reproducible FAIR dataset analysis integrating Croissant's modern data packaging.